# Pharmacy Demand Forecasting

## Project Overview

Forecasts **daily pharmacy prescription volume** over a 14-day horizon. Synthetic data spans 2 years with weekly patterns, flu-season demand, and monthly refill cycles.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily prescription counts, predict the next 14 days. Accurate forecasts help pharmacies manage drug inventory, staffing, and wait times.

## Why This Project Matters

Pharmacy demand forecasting prevents stockouts of critical medications and reduces patient wait times. It also optimizes pharmacist scheduling and inventory carrying costs.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily prescription counts
- Weekly pattern (higher weekdays, lower weekends)
- Monthly refill cycle (surge at month start)
- Flu season increase (Dec-Mar, more antibiotics/antivirals)
- Slight growth trend

| Column | Description |
|--------|-------------|
| `date` | Date |
| `prescriptions` | Daily prescription fill count |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'prescriptions'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

base = 250 + 0.1 * t
weekly = np.zeros(N_POINTS)
for i in range(N_POINTS):
    dow = dates[i].dayofweek
    if dow <= 4: weekly[i] = 40
    else: weekly[i] = -80  # much lower weekends

# Monthly refill cycle (first week of month)
monthly = np.zeros(N_POINTS)
for i in range(N_POINTS):
    d = dates[i].day
    if d <= 5: monthly[i] = 60

flu = np.zeros(N_POINTS)
for i in range(N_POINTS):
    m = dates[i].month
    if m in [12, 1, 2, 3]: flu[i] = 30

noise = np.random.normal(0, 20, N_POINTS)
prescriptions = base + weekly + monthly + flu + noise
prescriptions = np.maximum(prescriptions, 30).round(0).astype(int)

df = pd.DataFrame({'date': dates, 'prescriptions': prescriptions})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,prescriptions
0,2022-01-01,270
1,2022-01-02,257
2,2022-01-03,393
3,2022-01-04,411
4,2022-01-05,376


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('prescriptions Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("prescriptions Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

prescriptions Statistics:
count    730.000000
mean     311.458904
std       64.489036
min      158.000000
25%      268.000000
50%      325.000000
75%      356.000000
max      450.000000
Name: prescriptions, dtype: float64

CV: 0.207
Skewness: -0.456


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:       60.0 | RMSE:       65.6 | MAPE: 19.43%
Seasonal Naive (7)             | MAE:       13.4 | RMSE:       19.8 | MAPE:  4.65%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                           Adjusted R-Squared  R-Squared        RMSE  Time Taken
Model                                                                           
GaussianProcessRegressor           397.358892 -29.489146  336.465217    0.194802
KernelRidge                        380.350141 -28.180780  329.166777    0.054262
DummyRegressor                      23.512335  -0.731718   80.187402    0.016471
NuSVR                               19.083154  -0.391012   71.867570    0.041165
QuantileRegressor                   18.886024  -0.375848   71.474771    0.100747
MLPRegressor                        18.427711  -0.340593   70.553090    0.617505
SVR                                 16.884500  -0.221885   67.356986    0.036229
OrthogonalMatchingPursuit           11.232419   0.212891   54.061119    0.015433
XGBRegressor                         8.982392   0.385970   47.748791    1.159856
ExtraTreeRegressor                   8.320613   0.436876   45.726672    0.019142


## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: lgbm
FLAML (lgbm)                   | MAE:       27.6 | RMSE:       32.7 | MAPE:  7.76%
FLAML Test (lgbm)              | MAE:       28.9 | RMSE:       31.6 | MAPE:  7.88%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:       14.9 | RMSE:       17.4 | MAPE:  4.36%
SF AutoETS                     | MAE:       14.2 | RMSE:       19.2 | MAPE:  4.98%
SF SeasonalNaive               | MAE:       14.9 | RMSE:       21.6 | MAPE:  5.14%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
             model       MAE      RMSE      MAPE
Seasonal Naive (7) 13.357143 19.826029  4.653604
        SF AutoETS 14.192313 19.180115  4.983106
  SF SeasonalNaive 14.857142 21.603571  5.135864
      SF AutoARIMA 14.919954 17.377811  4.355300
      FLAML (lgbm) 27.633044 32.704715  7.761205
 FLAML Test (lgbm) 28.948774 31.580326  7.878012
Naive (Last Value) 60.000000 65.619030 19.431535

Top 3:
             model       MAE      RMSE     MAPE
Seasonal Naive (7) 13.357143 19.826029 4.653604
        SF AutoETS 14.192313 19.180115 4.983106
  SF SeasonalNaive 14.857142 21.603571 5.135864


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: 21.98, Std: 22.68


## Interpretation and Business Insight

- Weekday prescriptions are 40-50% higher than weekends
- Month-start refill cycles create predictable surges
- Flu season increases demand for antibiotics/antivirals
- Growth trend reflects population aging
- Lag features capture both weekly and monthly cycles

## Limitations

1. Synthetic — real pharmacy data varies by drug class, insurance type
2. No drug-level segmentation
3. No insurance/formulary change effects
4. Walk-in vs. mail-order not differentiated

## How to Improve This Project

1. Segment by drug category (chronic vs. acute)
2. Add flu surveillance data as features
3. Model individual high-value drug demand
4. Add insurance formulary change calendar
5. Use probabilistic forecasts for safety stock

## Production Considerations

- Daily retraining with pharmacy management system integration
- Drug-specific stock alerts based on forecasts
- Pharmacist scheduling optimization
- Monthly accuracy review with pharmacy managers

## Common Mistakes

1. Not accounting for month-start refill patterns
2. Treating weekends the same as weekdays
3. Ignoring flu season for antibiotic/antiviral inventory
4. Using aggregate forecasts for specific drug ordering

## Mini Challenge / Exercises

1. Add a month-day feature and measure improvement
2. Forecast separately for weekdays and weekends
3. Simulate a drug shortage scenario
4. Build a drug-specific reorder point system

## Final Summary / Key Takeaways

1. Pharmacy demand has strong weekly and monthly patterns
2. Month-start refill cycles are a unique and predictable pattern
3. Flu season increases demand for specific drug categories
4. 14-day forecasts enable inventory and staffing planning
5. Real-world deployment needs drug-level granularity

In [18]:
import json
metrics = {
    'project': 'Pharmacy Demand Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Pharmacy Demand Forecasting — Complete ===")

Metrics saved.

=== Pharmacy Demand Forecasting — Complete ===
